# CS 4354 HW3 Part 1: SQL Verification Agent (Google Colab)

This notebook runs the agent on a free Colab GPU, because `gpt-oss:20b` is too slow on a laptop without one.

**Before running:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells top to bottom.

It installs MySQL, loads the `soccerhub` database, creates a **read-only** database user for the agent, installs Ollama and `gpt-oss:20b`, runs the agent on Q4, Q6 and Q7, and lets you download the logs.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code

**Option A:** clone the GitHub repo (it must be public, or use a token).
**Option B:** if the repo isn't up yet, zip the project folder on your computer and upload it with the second cell.

In [ ]:
REPO_URL = "https://github.com/YOUR-USERNAME/cs4354-hw3-sql-agent.git"   # <- change this

!rm -rf /content/repo && git clone -q $REPO_URL /content/repo
%cd /content/repo
!ls

In [ ]:
# Option B: upload a .zip of the project instead (skip if Option A worked)
from google.colab import files
import zipfile, os, glob, shutil

uploaded = files.upload()
zip_name = next(iter(uploaded))
shutil.rmtree("/content/repo", ignore_errors=True)
with zipfile.ZipFile(zip_name) as z:
    z.extractall("/content/unzipped")
root = os.path.dirname(glob.glob("/content/unzipped/**/src/agent.py", recursive=True)[0])
shutil.move(os.path.dirname(root), "/content/repo")
%cd /content/repo
!ls

## 3. Install MySQL and load the soccerhub database

In [ ]:
!apt-get -qq update > /dev/null
!apt-get -qq install -y mysql-server zstd > /dev/null
!service mysql start
!mysql -u root < sql/soccerhub.sql
!mysql -u root -e "SELECT COUNT(*) AS teams FROM soccerhub.Team; SELECT COUNT(*) AS appearances FROM soccerhub.Appearance;"

## 4. Create the agent's read-only database user and the `.env` file

The agent connects as `agent_ro`, which only has `SELECT` permission, so even a bad query can't change data. The password is random and only exists in this Colab session.

In [ ]:
import secrets
ro_pass = secrets.token_hex(12)
!mysql -u root -e "DROP USER IF EXISTS 'agent_ro'@'%'; CREATE USER 'agent_ro'@'%' IDENTIFIED BY '{ro_pass}'; GRANT SELECT, SHOW VIEW ON soccerhub.* TO 'agent_ro'@'%'; FLUSH PRIVILEGES;"

with open(".env", "w") as f:
    f.write(f"""DB_USER=agent_ro
DB_PASS={ro_pass}
DB_HOST=127.0.0.1
DB_PORT=3306
DB_NAME=soccerhub
OLLAMA_MODEL=gpt-oss:20b
OLLAMA_URL=http://localhost:11434
NUM_CTX=16384
REASONING=medium
""")
print("agent_ro created, .env written")

## 5. Install Python packages

In [ ]:
!pip install -q -r requirements.txt

## 6. Install Ollama and download gpt-oss:20b (about 13 GB, takes a few minutes)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags"); print("Ollama is running"); break
    except Exception:
        time.sleep(1)

In [ ]:
!ollama pull gpt-oss:20b 2>&1 | tail -n 2

## 7. Quick checks before the real run

In [ ]:
# the tools can reach the database, and writes are refused
!cd src && python -c "import db_tools as d; print(d.execute_sql.invoke({'sql': 'SELECT COUNT(*) FROM Team'})); print(d.execute_sql.invoke({'sql': 'DELETE FROM Team'}))"

# the model answers (the first call also loads it onto the GPU)
!ollama run gpt-oss:20b "Reply with just: ready" 2>/dev/null
!ollama ps

## 8. Run the agent on Q4, Q6 and Q7

Each question can take a few minutes. The logs are saved after every question.

In [ ]:
!python src/agent.py

## 9. Look at the results

In [ ]:
import glob
from IPython.display import Markdown, display
report = sorted(glob.glob("logs/run_*_report.md"))[-1]
print(report)
display(Markdown(open(report).read()))

## 10. Download the logs (put them in the repo's `logs/` folder)

In [ ]:
!cd /content/repo && zip -qr /content/hw3_logs.zip logs
from google.colab import files
files.download("/content/hw3_logs.zip")